#Imports

In [66]:
import kagglehub

import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
from yellowbrick.cluster import KElbowVisualizer
from matplotlib.colors import ListedColormap
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.figure_factory as ff

from sklearn.preprocessing import StandardScaler, MinMaxScaler, LabelEncoder, OrdinalEncoder, OneHotEncoder
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import SimpleImputer, KNNImputer, IterativeImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
import scipy.stats as stats
from scipy.stats import norm, chi2
from statsmodels.stats.outliers_influence import variance_inflation_factor

from sklearn.model_selection import train_test_split,GridSearchCV, cross_val_score, StratifiedKFold

from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier, GradientBoostingClassifier, ExtraTreesClassifier, RandomForestRegressor, HistGradientBoostingClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from sklearn.naive_bayes import GaussianNB

from sklearn.metrics import accuracy_score, confusion_matrix, classification_report, mean_absolute_error, roc_auc_score, log_loss

import warnings
warnings.filterwarnings('ignore')

In [2]:
# Download latest version
path = kagglehub.dataset_download("redwankarimsony/heart-disease-data")

print("Path to dataset files:", path)

Using Colab cache for faster access to the 'heart-disease-data' dataset.
Path to dataset files: /kaggle/input/heart-disease-data


# Helping Functions

In [3]:
# Density Curve of Movies Duration
from scipy.stats import gaussian_kde

def hist(inds, x_axis_list, title, mode = 'line', ticks = 5, size = [4, 3]):

  fig = make_subplots(
      rows = size[0],
      cols = size[1],
      subplot_titles = [str(x_axis) for x_axis in x_axis_list],
      shared_yaxes = True,
      vertical_spacing = 0.1,
      horizontal_spacing = 0.05
  )

  for idx, x_axis in enumerate(x_axis_list):
    row = (idx // size[1]) + 1
    col = (idx % size[1]) + 1

    data = inds[x_axis].dropna()

    if mode == 'line':
      vals = np.linspace(data.min(), data.max(), 50)
      density = gaussian_kde(data)(vals)
      fig.add_trace(
          go.Scatter(
              x = vals,
              y = density,
              mode = 'lines',
              line = dict(width = 3, color = '#1f77b4'),
              fill = 'tozeroy',
              showlegend = False
          ),
          row = row,
          col = col
      )
    elif mode == 'hist':
      fig.add_trace(
          go.Histogram(
              x = data,
              nbinsx = 50,
              marker_color = "#636EFA",
              showlegend = False
          ),
          row = row,
          col = col
      )

  fig.update_layout(
      width = 400*size[1],
      height = 300*size[0],
      title = title,
      template = 'plotly_white',
      xaxis = dict(showgrid = True, gridcolor = 'lightgray', griddash = 'dash', dtick = ticks),
      yaxis = dict(showgrid = True, gridcolor = 'lightgray', griddash = 'dash')
  )

  fig.show()

In [4]:
def impute_mice(inds,
                predictor_num_cols,
                predictor_char_cols,
                target_cols,
                target_type = 0,
                test_size = 0.2,
                random_state = 42):
    """
    Imputation of missing values via MICE (Iterative Imputer + Classifier) (RandomForestClassifier).

    For Categorical data use target_type = 0, for Numeric data use target_type = 1

    Function takes data, encodes predictor character columns, encodes target columns if they are numeric.

    Then data is split to train and test and filler is trained, afterwards missing values are imputed.

    For Categorical data mean accuracy score is being derived.
    For Numeric data mean accuracy score is derived as average MAE.

    """
    data = inds[predictor_char_cols + predictor_num_cols + target_cols].copy()
    df_final = inds.copy()

    # Encoding values
    if len(predictor_char_cols) > 0:
      predictor_encoder = OrdinalEncoder()
      data[predictor_char_cols] = predictor_encoder.fit_transform(data[predictor_char_cols])

    if target_type == 0:
      data_encoder = OrdinalEncoder(handle_unknown = 'use_encoded_value', unknown_value = np.nan)
      data[target_cols] = data_encoder.fit_transform(data[target_cols])
      metrics = 'Accuracy'
      estimator = RandomForestClassifier(random_state = random_state)
      format = '.2%'
    elif target_type == 1:
      metrics = 'Root Mean Absolute Error'
      estimator = RandomForestRegressor(random_state = random_state)
      format = '.4f'
    else:
      print("Wrong value for type of target varibles")
      return

    complete_rows = data[target_cols].dropna().index

    # Check if there are enough records
    if len(complete_rows) > 10:
        np.random.seed(random_state)
        # Create test sample
        test_indices = np.random.choice(complete_rows,
                                        size = int(len(complete_rows) * test_size),
                                        replace = False)

        y_true = data.loc[test_indices, target_cols].copy()

        df_test_impute = data.copy()
        df_test_impute.loc[test_indices, target_cols] = np.nan

        test_imputer = IterativeImputer(estimator = estimator,
                                        max_iter = 10,
                                        random_state = random_state)

        df_test_filled = pd.DataFrame(test_imputer.fit_transform(df_test_impute),
                                      columns = data.columns)
        # Derive Accuracy scores
        acc_scores = []
        for col in target_cols:
          if target_type == 0:
            y_pred = df_test_filled.loc[test_indices, col].round()
            score = accuracy_score(y_true[col], y_pred)

          elif target_type == 1:
            score = mean_absolute_error(y_true[col], df_test_filled.loc[test_indices, col])

          acc_scores.append(score)

          print(f"MICE Imputation {metrics} for {col}: {score: {format}}")

        mean_score = np.mean(acc_scores)
        print(f"MICE Imputation Average {metrics}: {mean_score: {format}}")
        if target_type == 1:
          print(f"Root Mean Absolute Error is: {np.sqrt(mean_score): {format}}")
    else:
        print("Not enough records for estimation")

    # Final imputation of missing values
    final_imputer = IterativeImputer(estimator = estimator,
                                     max_iter = 10,
                                     random_state = random_state)

    df_filled = pd.DataFrame(final_imputer.fit_transform(data),
                                   columns = data.columns)

    # Preparing output dataset
    if target_type == 0:
      # Decode data if working with character target columns
      df_filled[target_cols] = df_filled[target_cols].round()

      df_final[target_cols] = data_encoder.inverse_transform(df_filled[target_cols])
    elif target_type == 1:
      df_final[target_cols] = df_filled[target_cols]

    return df_final

In [5]:
def cramers_v(x, y):
    confusion_matrix = pd.crosstab(x, y)
    chi2 = stats.chi2_contingency(confusion_matrix)[0]
    n = confusion_matrix.sum().sum()
    phi2 = chi2 / n
    r, k = confusion_matrix.shape
    phi2corr = max(0, phi2 - ((k-1)*(r-1))/(n-1))
    rcorr = r - ((r-1)**2)/(n-1)
    kcorr = k - ((k-1)**2)/(n-1)

    return np.sqrt(phi2corr / min((kcorr-1), (rcorr-1)))

In [6]:
def eta_squared(categorical, numeric):
    # Removes NaN safely for calculations
    df_temp = pd.DataFrame({'cat': categorical, 'num': numeric}).dropna()
    if len(df_temp) == 0: return 0
    groups = [group['num'].values for name, group in df_temp.groupby('cat')]
    # One-way ANOVA to find variances
    f_val, p_val = stats.f_oneway(*groups)
    if np.isnan(f_val): return 0
    # Convert F-value back to Eta
    n = len(df_temp)
    k = len(groups)
    eta2 = (f_val * (k - 1)) / (f_val * (k - 1) + (n - k))

    return np.sqrt(eta2)

In [7]:
def calculate_global_correlation(inds, all_cols, char_cols, num_cols):
    # Initialize empty matrix block
    matrix = pd.DataFrame(index=all_cols, columns=all_cols, dtype=float)

    for col1 in all_cols:
        for col2 in all_cols:
            if col1 == col2:
                matrix.loc[col1, col2] = 1.0
            # Condition A: Numeric vs Numeric
            elif col1 in num_cols and col2 in num_cols:
                matrix.loc[col1, col2] = inds[col1].corr(inds[col2], method='pearson')
            # Condition B: Categorical vs Categorical
            elif col1 in char_cols and col2 in char_cols:
                matrix.loc[col1, col2] = cramers_v(inds[col1], inds[col2])
            # Condition C: Categorical vs Numeric
            else:
                cat_col = col1 if col1 in char_cols else col2
                num_col = col2 if col2 in num_cols else col1
                matrix.loc[col1, col2] = eta_squared(inds[cat_col], inds[num_col])

    return matrix

In [8]:
def hosmer_lemeshow_test(y_true, y_prob, g=10):
    data_hl = pd.DataFrame({'y_true': y_true, 'y_prob': y_prob})
    data_hl['q'] = pd.qcut(data_hl['y_prob'], g, labels=False, duplicates='drop')

    observed = data_hl.groupby('q')['y_true'].agg(['sum', 'count'])
    observed.columns = ['obs_cases', 'total']
    observed['obs_non_cases'] = observed['total'] - observed['obs_cases']

    expected = data_hl.groupby('q')['y_prob'].agg(['sum', 'count'])
    expected.columns = ['exp_prob', 'total']
    expected['exp_cases'] = expected['exp_prob']
    expected['exp_non_cases'] = expected['total'] - expected['exp_cases']

    # Chi2
    chi2_val = 0
    for index, row in observed.iterrows():
        exp_row = expected.loc[index]
        o_c, e_c = row['obs_cases'], exp_row['exp_cases']
        o_nc, e_nc = row['obs_non_cases'], exp_row['exp_non_cases']

        chi2_val += ((o_c - e_c)**2 / e_c) if e_c > 0 else 0
        chi2_val += ((o_nc - e_nc)**2 / e_nc) if e_nc > 0 else 0

    df = len(observed) - 2
    p_value = 1 - chi2.cdf(chi2_val, df)

    print(f"Chisq: {chi2_val:.4f}")
    print(f"p-value: {p_value:.4f}")
    if p_value > 0.05:
      print("H_0 can be accepted and model works well")
    else:
      print("Model has issues")

    return chi2_val, p_value

In [9]:
def build_logistic_regression(dataset, num_features, char_features):
  """
  Building Scikit-Learn Logistic Regression and performs tests with statistical report:
    - Estimates,
    - p-values,
    - Pseudo R^2,
    - VIF
    - Hosmer-Lemeshow test
  """
  # Predictors and Dependend Variable
  all_features = num_features + char_features
  X = dataset[all_features].copy()
  y = dataset['num'].copy()

  # Standard Scaler + One-Hot Encoder
  preprocessor = ColumnTransformer(
      transformers=[
          ('num', StandardScaler(), num_features),
          ('cat', OneHotEncoder(drop='first', handle_unknown='ignore'), char_features)
      ]
  )

  # Transform Data
  X_trans = preprocessor.fit_transform(X)
  model = LogisticRegression(penalty=None, random_state=42, max_iter=1000)
  model.fit(X_trans, y)

  # Feature's names
  cat_encoder = preprocessor.named_transformers_['cat']
  encoded_cat_features = cat_encoder.get_feature_names_out(char_features).tolist()
  feature_names = num_features + encoded_cat_features

  # Estimates, Intercept
  estimates = model.coef_[0]
  intercept = model.intercept_[0]

  # Design matrix with Intercept for VIF and errors
  X_design = np.hstack([np.ones((X_trans.shape[0], 1)), X_trans])
  all_estimates = np.insert(estimates, 0, intercept)
  all_feature_names = ['Intercept'] + feature_names

  # Derive predicted probabilities
  y_proba = np.clip(model.predict_proba(X_trans)[:, 1], 1e-15, 1 - 1e-15)
  W = np.diag(y_proba * (1 - y_proba))

  print("=========================================================================")
  print("                LOGISTIC REGRESSION STATISTICAL REPORT                   ")
  print("=========================================================================")

  # Estimate, Standard Error, p-values
  try:
      cov_matrix = np.linalg.inv(X_design.T @ W @ X_design)
      standard_errors = np.sqrt(np.diag(cov_matrix))

      # Z-score, p-values
      z_scores = all_estimates / standard_errors
      p_values = [2 * (1 - norm.cdf(np.abs(z))) for z in z_scores]

      summary_df = pd.DataFrame({
          'Feature': all_feature_names,
          'Estimate (Coef)': all_estimates,
          'Std Error': standard_errors,
          'z-score': z_scores,
          'p-value': p_values
      })

      # Formatting
      summary_df['Estimate (Coef)'] = summary_df['Estimate (Coef)'].map('{:.4f}'.format)
      summary_df['Std Error'] = summary_df['Std Error'].map('{:.4f}'.format)
      summary_df['p-value'] = summary_df['p-value'].map(lambda x: '<0.0001' if x < 0.0001 else f'{x:.4f}')

      print(summary_df.to_string(index = False))

  except np.linalg.LinAlgError:
      print("\n[Error]: Issue with matrix, derivation of p-value is not possible.")

  print("\n=========================================================================")
  print("                      MODEL DIAGNOSTICS & METRICS                        ")
  print("=========================================================================")

  # McFadden Pseudo R^2
  log_like_model = np.sum(y * np.log(y_proba) + (1 - y) * np.log(1 - y_proba))
  p_null = np.mean(y)
  log_like_null = np.sum(y * np.log(p_null) + (1 - y) * np.log(1 - p_null))
  pseudo_r2 = 1 - (log_like_model / log_like_null)

  print(f"McFadden Pseudo R^2: {pseudo_r2:.4f}")
  print("-------------------------------------------------------------------------")

  # VIF
  vif_data = pd.DataFrame()
  vif_data["Feature"] = all_feature_names
  vif_data["VIF"] = [variance_inflation_factor(X_design, i) for i in range(X_design.shape[1])]
  vif_data["VIF"] = vif_data["VIF"].map('{:.4f}'.format)

  print("Variance Inflation Factor (VIF):")
  print(vif_data.to_string(index = False))
  print("-------------------------------------------------------------------------")

  # Hosmer-Lemeshow
  print("Hosmer-Lemeshow Goodness-of-Fit Test:")
  chi2_stat, hl_p_value = hosmer_lemeshow_test(y, y_proba)
  print("=========================================================================")

  return model, preprocessor, all_feature_names

In [44]:
def multinomial_logistic_regression(X_train, y_train, X_test, y_test, feature_names = feature_names, all_features_intercept = all_features_intercept, penalty = 'l2', C = 1.0):
    """
    Building Scikit-Learn Multinomial Logistic Regression
    and performs tests with statistical report:
      - Estimates for each class,
      - p-values,
      - Pseudo R^2,
      - VIF
    """
    # Check if data is sparsed
    X_tr = X_train.toarray() if hasattr(X_train, "toarray") else np.asarray(X_train)
    X_te = X_test.toarray() if hasattr(X_test, "toarray") else np.asarray(X_test)

    y_tr = np.asarray(y_train).astype(int)
    y_te = np.asarray(y_test).astype(int)

    # Model fitting
    model = LogisticRegression(
        penalty=penalty,
        C=C,
        solver='newton-cg',
        random_state=42,
        max_iter=2000
    )

    model.fit(X_tr, y_tr)

    # Model training
    X_design = np.hstack([np.ones((X_tr.shape[0], 1)), X_tr])
    y_proba_tr = np.clip(model.predict_proba(X_tr), 1e-15, 1 - 1e-15)
    is_binary = len(model.classes_) == 2

    print("=========================================================================")
    print("         TRAINING PHASE: MULTINOMIAL LOGISTIC STATISTICAL REPORT        ")
    print("=========================================================================")

    for idx, cls in enumerate(model.classes_):
        print(f"\n--- Coefficients for Class {cls} vs All Other Classes ---")

        if is_binary:
            estimates = model.coef_[0] if idx == 1 else - model.coef_[0]
            intercept = model.intercept_[0] if idx == 1 else - model.intercept_[0]
        else:
            estimates = model.coef_[idx]
            intercept = model.intercept_[idx]

        all_estimates = np.insert(estimates, 0, intercept)

        w_vector = y_proba_tr[:, idx] * (1.0 - y_proba_tr[:, idx])
        X_W = X_design * w_vector[:, np.newaxis]
        X_W_X = X_design.T @ X_W

        try:
            if penalty is None or penalty == 'none':
                cov_matrix = np.linalg.inv(X_W_X)
            elif penalty == 'l2':
                lam = 1.0 / C
                penalty_matrix = np.diag([0.0] + [lam] * X_tr.shape[1])
                inv_reg = np.linalg.inv(X_W_X + penalty_matrix)
                cov_matrix = inv_reg @ X_W_X @ inv_reg
            else:
                raise ValueError("Approximated errors only supported for L2 or None penalties.")

            standard_errors = np.sqrt(np.diag(cov_matrix))
            z_scores = all_estimates / standard_errors
            p_values = [2 * (1 - norm.cdf(np.abs(z))) for z in z_scores]

            summary_df = pd.DataFrame({
                'Feature': all_features_intercept,
                'Estimate (Coef)': all_estimates,
                'Std Error': standard_errors,
                'z-score': z_scores,
                'p-value': p_values
            })

            summary_df['Estimate (Coef)'] = summary_df['Estimate (Coef)'].map('{:.4f}'.format)
            summary_df['Std Error'] = summary_df['Std Error'].map('{:.4f}'.format)
            summary_df['p-value'] = summary_df['p-value'].map(lambda x: '<0.0001' if x < 0.0001 else f'{x:.4f}')
            print(summary_df.to_string(index = False))

        except np.linalg.LinAlgError:
            print(f"\n[Error]: Issue with matrix for class {cls}, derivation of p-value is not possible.")

    print("\n-------------------------------------------------------------------------")
    print("Multinomial McFadden Pseudo R^2 (Training Fit):")
    log_like_model = sum(np.sum((y_tr == cls) * np.log(y_proba_tr[:, idx])) for idx, cls in enumerate(model.classes_))
    class_frequencies = np.bincount(y_tr) / len(y_tr)
    log_like_null = sum(np.sum((y_tr == cls) * np.log(class_frequencies[idx])) for idx, cls in enumerate(model.classes_))
    pseudo_r2 = 1 - (log_like_model / log_like_null)
    print(f"  Pseudo R^2: {pseudo_r2:.4f}")

    print("\nVariance Inflation Factor (VIF):")
    vif_data = pd.DataFrame()
    vif_data["Feature"] = feature_names
    vif_data["VIF"] = [variance_inflation_factor(X_tr, i) for i in range(X_tr.shape[1])]
    vif_data["VIF"] = vif_data["VIF"].map('{:.4f}'.format)
    print(vif_data.to_string(index = False))

    # Model Evaluation on the test data
    print("\n=========================================================================")
    print("                      TEST PHASE: EVALUATION METRICS                     ")
    print("=========================================================================")

    y_pred = model.predict(X_te)
    y_proba_te = model.predict_proba(X_te)

    test_accuracy = accuracy_score(y_te, y_pred)
    test_loss = log_loss(y_te, y_proba_te)

    print(f"Test Accuracy: {test_accuracy:.4f}")
    print(f"Test Log Loss: {test_loss:.4f}")
    print("-------------------------------------------------------------------------")
    print("Detailed Test Classification Matrix Metrics:")
    print(classification_report(y_te, y_pred))
    print("=========================================================================")

    return model

In [45]:
def knn_classifier(X_train, y_train, X_test, y_test, n_neighbors = 5, weights = 'uniform'):
    """
    Scikit-Learn K-Nearest Neighbors Classifier Model
    """
    # Check if data is sparsed
    X_tr = X_train.toarray() if hasattr(X_train, "toarray") else np.asarray(X_train)
    X_te = X_test.toarray() if hasattr(X_test, "toarray") else np.asarray(X_test)

    y_tr = np.asarray(y_train).astype(int)
    y_te = np.asarray(y_test).astype(int)

    # Model fitting
    model = KNeighborsClassifier(
        n_neighbors = n_neighbors,
        weights = weights,
        metric = 'minkowski',
        p = 2
    )

    model.fit(X_tr, y_tr)

    # Model training
    y_pred = model.predict(X_te)
    y_proba = model.predict_proba(X_te)

    print("=========================================================================")
    print("                      KNN CLASSIFIER EVALUATION REPORT                   ")
    print("=========================================================================")

    # Model Evaluation
    class_frequencies = np.bincount(y_tr) / len(y_tr)
    baseline_proba = np.tile(class_frequencies, (len(y_te), 1))

    knn_loss = log_loss(y_te, y_proba, labels = model.classes_)
    baseline_loss = log_loss(y_te, baseline_proba, labels = model.classes_)

    log_loss_r2 = 1 - (knn_loss / baseline_loss)
    test_accuracy = accuracy_score(y_te, y_pred)

    print(f"Test Accuracy: {test_accuracy:.4f}")
    print(f"Test Log Loss: {knn_loss:.4f}")
    print(f"Derived Log Loss R^2: {log_loss_r2:.4f}")
    print("-------------------------------------------------------------------------")
    print("Detailed Classification Matrix Metrics:")
    print(classification_report(y_te, y_pred))
    print("=========================================================================")

    return model

In [58]:
def gradient_boosting_classifier(X_train, y_train, X_test, y_test, learning_rate = 0.1, max_iter = 100):
    """
    Scikit-Learn Gradient Boosting Classifier
    """
    # Check if data is sparsed
    X_tr = X_train.toarray() if hasattr(X_train, "toarray") else np.asarray(X_train)
    X_te = X_test.toarray() if hasattr(X_test, "toarray") else np.asarray(X_test)

    y_tr = np.asarray(y_train).astype(int)
    y_te = np.asarray(y_test).astype(int)

    # Model fitting
    model = HistGradientBoostingClassifier(
        learning_rate = learning_rate,
        max_iter = max_iter,
        random_state = 42
    )

    model.fit(X_tr, y_tr)

    # Model training
    y_pred = model.predict(X_te)
    y_proba = np.clip(model.predict_proba(X_te), 1e-15, 1 - 1e-15)

    print("=========================================================================")
    print("                 GRADIENT BOOSTING EVALUATION REPORT                    ")
    print("=========================================================================")

    # Model Evaluation
    class_frequencies = np.bincount(y_tr) / len(y_tr)
    baseline_proba = np.tile(class_frequencies, (len(y_te), 1))

    gb_loss = log_loss(y_te, y_proba, labels = model.classes_)
    baseline_loss = log_loss(y_te, baseline_proba, labels = model.classes_)

    log_loss_r2 = 1 - (gb_loss / baseline_loss)
    test_accuracy = accuracy_score(y_te, y_pred)

    print(f"Test Accuracy: {test_accuracy:.4f}")
    print(f"Test Log Loss: {gb_loss:.4f}")
    print(f"Derived Log Loss R^2: {log_loss_r2:.4f}")
    print("-------------------------------------------------------------------------")
    print("Detailed Classification Matrix Metrics:")
    print(classification_report(y_te, y_pred))
    print("=========================================================================")

    return model

In [60]:
def decision_tree_classifier(X_train, y_train, X_test, y_test, max_depth = None, criterion = 'gini'):
    """
    Building Scikit-Learn Decision Tree Classifier
    """
    # Check if data is sparsed
    X_tr = X_train.toarray() if hasattr(X_train, "toarray") else np.asarray(X_train)
    X_te = X_test.toarray() if hasattr(X_test, "toarray") else np.asarray(X_test)

    y_tr = np.asarray(y_train).astype(int)
    y_te = np.asarray(y_test).astype(int)

    # Model fitting
    model = DecisionTreeClassifier(
        criterion = criterion,
        max_depth = max_depth,
        random_state = 42
    )

    model.fit(X_tr, y_tr)

    # Model training
    y_pred = model.predict(X_te)
    y_proba = np.clip(model.predict_proba(X_te), 1e-15, 1 - 1e-15)

    print("=========================================================================")
    print("                   DECISION TREE EVALUATION REPORT                       ")
    print("=========================================================================")

    # Model Evaluation
    class_frequencies = np.bincount(y_tr) / len(y_tr)
    baseline_proba = np.tile(class_frequencies, (len(y_te), 1))

    dt_loss = log_loss(y_te, y_proba, labels = model.classes_)
    baseline_loss = log_loss(y_te, baseline_proba, labels = model.classes_)

    log_loss_r2 = 1 - (dt_loss / baseline_loss)
    test_accuracy = accuracy_score(y_te, y_pred)

    print(f"Test Accuracy: {test_accuracy:.4f}")
    print(f"Test Log Loss: {dt_loss:.4f}")
    print(f"Derived Log Loss R^2: {log_loss_r2:.4f}")
    print("-------------------------------------------------------------------------")
    print("Detailed Classification Matrix Metrics:")
    print(classification_report(y_te, y_pred))
    print("=========================================================================")

    return model

In [61]:
def random_forest_classifier(X_train, y_train, X_test, y_test, n_estimators = 100, max_depth = None):
    """
    Building Scikit-Learn Random Forest Classifier
    """
    # Check if data is sparsed
    X_tr = X_train.toarray() if hasattr(X_train, "toarray") else np.asarray(X_train)
    X_te = X_test.toarray() if hasattr(X_test, "toarray") else np.asarray(X_test)

    y_tr = np.asarray(y_train).astype(int)
    y_te = np.asarray(y_test).astype(int)

    # Model fitting
    model = RandomForestClassifier(
        n_estimators = n_estimators,
        max_depth = max_depth,
        random_state = 42,
        n_jobs = -1
    )

    model.fit(X_tr, y_tr)

    # Model training
    y_pred = model.predict(X_te)
    y_proba = np.clip(model.predict_proba(X_te), 1e-15, 1 - 1e-15)

    print("=========================================================================")
    print("                   RANDOM FOREST EVALUATION REPORT                       ")
    print("=========================================================================")

    # Model Evaluation
    class_frequencies = np.bincount(y_tr) / len(y_tr)
    baseline_proba = np.tile(class_frequencies, (len(y_te), 1))

    rf_loss = log_loss(y_te, y_proba, labels = model.classes_)
    baseline_loss = log_loss(y_te, baseline_proba, labels = model.classes_)

    log_loss_r2 = 1 - (rf_loss / baseline_loss)
    test_accuracy = accuracy_score(y_te, y_pred)

    print(f"Test Accuracy: {test_accuracy:.4f}")
    print(f"Test Log Loss: {rf_loss:.4f}")
    print(f"Derived Log Loss R^2: {log_loss_r2:.4f}")
    print("-------------------------------------------------------------------------")
    print("Detailed Classification Matrix Metrics:")
    print(classification_report(y_te, y_pred))
    print("=========================================================================")

    return model

In [62]:
def xgboost_classifier(X_train, y_train, X_test, y_test, learning_rate = 0.1, n_estimators = 100, max_depth = 3):
    """
    Building XGBoost Classifier
    """
    # Check if data is sparsed
    X_tr = X_train.toarray() if hasattr(X_train, "toarray") else np.asarray(X_train)
    X_te = X_test.toarray() if hasattr(X_test, "toarray") else np.asarray(X_test)

    y_tr = np.asarray(y_train).astype(int)
    y_te = np.asarray(y_test).astype(int)

    # Model fitting
    model = XGBClassifier(
        n_estimators = n_estimators,
        learning_rate = learning_rate,
        max_depth = max_depth,
        objective = 'multi:softprob',
        random_state = 42,
        eval_metric = 'mlogloss',
        n_jobs = -1
    )

    model.fit(X_tr, y_tr)

    # Model training
    y_pred = model.predict(X_te)
    y_proba = np.clip(model.predict_proba(X_te), 1e-15, 1 - 1e-15)

    print("=========================================================================")
    print("                      XGBOOST EVALUATION REPORT                          ")
    print("=========================================================================")

    # Model Evaluation
    class_frequencies = np.bincount(y_tr) / len(y_tr)
    baseline_proba = np.tile(class_frequencies, (len(y_te), 1))

    xgb_loss = log_loss(y_te, y_proba, labels=model.classes_)
    baseline_loss = log_loss(y_te, baseline_proba, labels=model.classes_)

    log_loss_r2 = 1 - (xgb_loss / baseline_loss)
    test_accuracy = accuracy_score(y_te, y_pred)

    print(f"Test Accuracy: {test_accuracy:.4f}")
    print(f"Test Log Loss: {xgb_loss:.4f}")
    print(f"Derived Log Loss R^2: {log_loss_r2:.4f}")
    print("-------------------------------------------------------------------------")
    print("Detailed Classification Matrix Metrics:")
    print(classification_report(y_te, y_pred))
    print("=========================================================================")

    return model

In [64]:
def adaboost_classifier(X_train, y_train, X_test, y_test, n_estimators = 50, learning_rate = 1.0):
    """
    Building Scikit-Learn AdaBoost Classifier
    """
    # Check if data is sparsed
    X_tr = X_train.toarray() if hasattr(X_train, "toarray") else np.asarray(X_train)
    X_te = X_test.toarray() if hasattr(X_test, "toarray") else np.asarray(X_test)

    y_tr = np.asarray(y_train).astype(int)
    y_te = np.asarray(y_test).astype(int)

    # Model fitting
    model = AdaBoostClassifier(
        n_estimators=n_estimators,
        learning_rate=learning_rate,
        algorithm='SAMME',
        random_state=42
    )
    model.fit(X_tr, y_tr)

    # Model training
    y_pred = model.predict(X_te)
    y_proba = np.clip(model.predict_proba(X_te), 1e-15, 1 - 1e-15)

    print("=========================================================================")
    print("                      ADABOOST EVALUATION REPORT                         ")
    print("=========================================================================")

    # Model Evaluation
    class_frequencies = np.bincount(y_tr) / len(y_tr)
    baseline_proba = np.tile(class_frequencies, (len(y_te), 1))

    ada_loss = log_loss(y_te, y_proba, labels=model.classes_)
    baseline_loss = log_loss(y_te, baseline_proba, labels=model.classes_)

    log_loss_r2 = 1 - (ada_loss / baseline_loss)
    test_accuracy = accuracy_score(y_te, y_pred)

    print(f"Test Accuracy: {test_accuracy:.4f}")
    print(f"Test Log Loss: {ada_loss:.4f}")
    print(f"Derived Log Loss R^2: {log_loss_r2:.4f}")
    print("-------------------------------------------------------------------------")
    print("Detailed Classification Matrix Metrics:")
    print(classification_report(y_te, y_pred))
    print("=========================================================================")

    return model

In [69]:
def model_tuning(X_train, y_train, X_test, y_test, feature_names, all_features_intercept):
    """
    Performs GridSearchCV fine-tuning for all 7 models, outputs independent
    metric summaries, tracks CV/Test scores, and determines the best model.
    """
    # 1. Handle potential sparse arrays safely
    X_tr = X_train.toarray() if hasattr(X_train, "toarray") else np.asarray(X_train)
    X_te = X_test.toarray() if hasattr(X_test, "toarray") else np.asarray(X_test)

    y_tr = np.asarray(y_train).astype(int)
    y_te = np.asarray(y_test).astype(int)

    # 2. Define Hyperparameter Grids for Grid Search
    model_configs = {
        "Logistic Regression": {
            "model": LogisticRegression(
                solver = 'newton-cg',
                random_state = 42,
                max_iter = 2000
            ),
            "param_grid": {
                'penalty': ['l2', None],
                'C': [0.1, 1.0, 10.0]
            }
        },
        "K-Nearest Neighbors": {
            "model": KNeighborsClassifier(),
            "param_grid": {
                'n_neighbors': [3, 5, 7, 11],
                'weights': ['uniform', 'distance']
            }
        },
        "Gradient Boosting": {
            "model": HistGradientBoostingClassifier(
                random_state = 42
            ),
            "param_grid": {
                'learning_rate': [0.01, 0.05, 0.1],
                'max_iter': [50, 100, 150]
            }
        },
        "Decision Tree": {
            "model": DecisionTreeClassifier(
                random_state = 42
            ),
            "param_grid": {
                'max_depth': [3, 4, 5, 6],
                'criterion': ['gini', 'entropy']
            }
        },
        "Random Forest": {
            "model": RandomForestClassifier(
                random_state = 42,
                n_jobs = -1
            ),
            "param_grid": {
                'n_estimators': [50, 100, 150],
                'max_depth': [4, 6, 8]
            }
        },
        "XGBoost": {
            "model": XGBClassifier(
                objective = 'multi:softprob',
                random_state = 42,
                eval_metric = 'mlogloss',
                n_jobs = -1
            ),
            "param_grid": {
                'n_estimators': [50, 100],
                'learning_rate': [0.01, 0.05, 0.1],
                'max_depth': [3, 4, 5]
            }
        },
        "AdaBoost": {
            "model": AdaBoostClassifier(
                algorithm = 'SAMME',
                random_state = 42
            ),
            "param_grid": {
                'n_estimators': [50, 100, 150],
                'learning_rate': [0.05, 0.1, 0.5]
            }
        }
    }

    # 3. Calculate Baseline Null Log Loss for the Log Loss R^2 metric
    class_frequencies = np.bincount(y_tr) / len(y_tr)
    baseline_proba = np.tile(class_frequencies, (len(y_te), 1))
    baseline_loss = log_loss(y_te, baseline_proba, labels = np.unique(y_tr))

    # Cross-validation engine setup
    cv_strategy = StratifiedKFold(n_splits = 5, shuffle = True, random_state = 42)

    results_list = []
    best_estimators = {}

    print("=========================================================================")
    print("                 STARTING HYPERPARAMETER FINE-TUNING                     ")
    print("=========================================================================")

    # 4. Loop over every model configuration
    for name, config in model_configs.items():
        print(f"\nTuning {name} via GridSearchCV...")

        # Grid Search Setup optimizing for Accuracy
        grid_search = GridSearchCV(
            estimator = config["model"],
            param_grid = config["param_grid"],
            cv = cv_strategy,
            scoring = 'accuracy',
            n_jobs = -1
        )

        grid_search.fit(X_tr, y_tr)

        # Extract best parameters and optimal estimator
        best_model = grid_search.best_estimator_
        best_estimators[name] = best_model

        cv_mean_acc = grid_search.best_score_
        # Get standard deviation for the optimal hyperparameter index
        cv_std_acc = grid_search.cv_results_['std_test_score'][grid_search.best_index_]

        # Test Set Predictions using Optimized Model
        y_pred = best_model.predict(X_te)
        y_proba = np.clip(best_model.predict_proba(X_te), 1e-15, 1 - 1e-15)

        test_accuracy = accuracy_score(y_te, y_pred)
        test_loss = log_loss(y_te, y_proba, labels = best_model.classes_)
        log_loss_r2 = 1 - (test_loss / baseline_loss)

        # Save evaluation data footprint
        results_list.append({
            "Model Name": name,
            "Best Params": grid_search.best_params_,
            "CV Accuracy (Mean)": cv_mean_acc,
            "CV Accuracy (Std)": cv_std_acc,
            "Test Accuracy": test_accuracy,
            "Test Log Loss": test_loss,
            "Log Loss R^2": log_loss_r2
        })

        # PRINT INDIVIDUAL MODEL PROFILE
        print(f"---------------------------------------------------------")
        print(f" METRICS SUMMARY: {name.upper()}")
        print(f"---------------------------------------------------------")
        print(f"  Optimized Hyperparameters : {grid_search.best_params_}")
        print(f"  5-Fold CV Accuracy (Mean) : {cv_mean_acc:.4f} (±{cv_std_acc:.4f})")
        print(f"  Test Accuracy             : {test_accuracy:.4f}")
        print(f"  Test Log Loss             : {test_loss:.4f}")
        print(f"  Log Loss R^2 Score        : {log_loss_r2:.4f}")
        print(f"\nDetailed Classification Matrix (Test Data):")
        print(classification_report(y_te, y_pred))
        print(f"=========================================================")

    # 5. Process and Display Final Leaderboard Dataframe
    leaderboard = pd.DataFrame(results_list)

    # Primary sorting: Best Test Accuracy. Secondary sorting: Best Log Loss R^2.
    leaderboard = leaderboard.sort_values(by = ["Test Accuracy", "Log Loss R^2"], ascending = [False, False]).reset_index(drop=True)

    print("\n=========================================================================")
    print("                 FINAL TUNED PERFORMANCE LEADERBOARD                     ")
    print("=========================================================================")

    formatted_df = leaderboard.copy()
    formatted_df["CV Accuracy (Mean)"] = formatted_df["CV Accuracy (Mean)"].map('{:.4f}'.format)
    formatted_df["CV Accuracy (Std)"] = formatted_df["CV Accuracy (Std)"].map('{:.4f}'.format)
    formatted_df["Test Accuracy"] = formatted_df["Test Accuracy"].map('{:.4f}'.format)
    formatted_df["Test Log Loss"] = formatted_df["Test Log Loss"].map('{:.4f}'.format)
    formatted_df["Log Loss R^2"] = formatted_df["Log Loss R^2"].map('{:.4f}'.format)

    # Hide the dictionary configuration for cleaner terminal display out
    print(formatted_df.drop(columns=["Best Params"]).to_string(index=False))
    print("=========================================================================")

    # 6. Announce the Winner
    best_model_row = leaderboard.iloc[0]
    print(f"\n🏆 WINNING PERFORMANCE SELECTION 🏆")
    print(f"The best performing model is **{best_model_row['Model Name']}**.")
    print(f"  - Final Test Accuracy: {best_model_row['Test Accuracy']:.4f}")
    print(f"  - Log Loss R^2 Score: {best_model_row['Log Loss R^2']:.4f}")
    print(f"  - Chosen Tuning Settings: {best_model_row['Best Params']}")
    print("=========================================================================")

    return best_estimators, leaderboard

# Data Cleaning&Investigation

In [12]:
init_df = pd.read_csv(path + "/heart_disease_uci.csv")

##General Information

In [13]:
init_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 920 entries, 0 to 919
Data columns (total 16 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   id        920 non-null    int64  
 1   age       920 non-null    int64  
 2   sex       920 non-null    object 
 3   dataset   920 non-null    object 
 4   cp        920 non-null    object 
 5   trestbps  861 non-null    float64
 6   chol      890 non-null    float64
 7   fbs       830 non-null    object 
 8   restecg   918 non-null    object 
 9   thalch    865 non-null    float64
 10  exang     865 non-null    object 
 11  oldpeak   858 non-null    float64
 12  slope     611 non-null    object 
 13  ca        309 non-null    float64
 14  thal      434 non-null    object 
 15  num       920 non-null    int64  
dtypes: float64(5), int64(3), object(8)
memory usage: 115.1+ KB


In [14]:
# Print ratio of missing data in the columns
for i in init_df.columns:
    null_rate = round(init_df[i].isna().sum() / len(init_df) * 100, 1)
    if null_rate > 0 :
        print(f"{i} null rate: {null_rate}%")
    else:
      print(f"{i} does not contain missing values")

id does not contain missing values
age does not contain missing values
sex does not contain missing values
dataset does not contain missing values
cp does not contain missing values
trestbps null rate: 6.4%
chol null rate: 3.3%
fbs null rate: 9.8%
restecg null rate: 0.2%
thalch null rate: 6.0%
exang null rate: 6.0%
oldpeak null rate: 6.7%
slope null rate: 33.6%
ca null rate: 66.4%
thal null rate: 52.8%
num does not contain missing values


In [15]:
init_df.describe()

,id,age,trestbps,chol,thalch,oldpeak,ca,num
count,920.000000,920.000000,861.000000,890.000000,865.000000,858.000000,309.000000,920.000000
mean,460.500000,53.510870,132.132404,199.130337,137.545665,0.878788,0.676375,0.995652
std,265.725422,9.424685,19.066070,110.780810,25.926276,1.091226,0.935653,1.142693
min,1.000000,28.000000,0.000000,0.000000,60.000000,-2.600000,0.000000,0.000000
25%,230.750000,47.000000,120.000000,175.000000,120.000000,0.000000,0.000000,0.000000
50%,460.500000,54.000000,130.000000,223.000000,140.000000,0.500000,0.000000,1.000000
75%,690.250000,60.000000,140.000000,268.000000,157.000000,1.500000,1.000000,2.000000
max,920.000000,77.000000,200.000000,603.000000,202.000000,6.200000,3.000000,4.000000


##Distributions

In [16]:
hist(inds = init_df, x_axis_list = ['age', 'trestbps', 'chol', 'thalch', 'oldpeak', 'ca', 'num'], title = 'Histograms of distributions of numeric variables', size = [3, 3])

In [17]:
hist(inds = init_df, x_axis_list = ['sex', 'dataset', 'cp', 'fbs', 'restecg', 'exang', 'slope', 'thal'], title = 'Histograms of distributions of character variables', mode = 'hist', size = [3, 3])

As we can see from the figures above, not all numeric values are distributed normally

##Handling Missing Data

In [18]:
df_filled = init_df.copy()

###Numeric Variables

####Simple Imputer

In [19]:
simple_imputer_mean = SimpleImputer(strategy = 'mean')
simple_imputer_median = SimpleImputer(strategy = 'median')

#####Mean + Median for categorical columns

In [20]:
df_filled['oldpeak'] = simple_imputer_mean.fit_transform(init_df[['oldpeak']])
df_filled['ca'] = simple_imputer_median.fit_transform(init_df[['ca']])

In [21]:
hist(inds = df_filled, x_axis_list = ['oldpeak', 'ca'], title = 'Histograms of distributions of numeric variables', size = [1, 2])

#####Median

In [22]:
df_filled['oldpeak'] = simple_imputer_median.fit_transform(init_df[['oldpeak']])

In [23]:
hist(inds = df_filled, x_axis_list = ['oldpeak'], title = 'Histograms of distributions of numeric variables', size = [1, 1])

####Iterative Imputer

In [24]:
iterative_imputer = IterativeImputer(max_iter = 15, random_state = 42)

In [25]:
df_filled['oldpeak'] = iterative_imputer.fit_transform(init_df[['oldpeak']])

In [26]:
hist(inds = df_filled, x_axis_list = ['oldpeak'], title = 'Histograms of distributions of numeric variables', size = [1, 1])

###Character Variables

####Most Frequent Value

In [27]:
char_imputer_mode = SimpleImputer(strategy = 'most_frequent')

In [28]:
targets = ['fbs', 'restecg', 'exang', 'slope', 'thal']
df_filled[targets] = char_imputer_mode.fit_transform(init_df[targets])

In [29]:
hist(inds = df_filled, x_axis_list = ['fbs', 'restecg', 'exang', 'slope', 'thal'], title = 'Histograms of distributions of character variables', mode = 'hist', size = [2, 3])

####KNN

In [30]:
df_pre_knn_mice = init_df[targets + ['age', 'sex', 'dataset']].copy()

In [31]:
predictor_encoder = OrdinalEncoder()
data_encoder = OrdinalEncoder(handle_unknown = 'use_encoded_value', unknown_value = np.nan)
df_pre_knn_mice[['sex', 'dataset']] = predictor_encoder.fit_transform(df_pre_knn_mice[['sex', 'dataset']])
df_pre_knn_mice[targets] = data_encoder.fit_transform(df_pre_knn_mice[targets])

knn_imputer = KNNImputer(n_neighbors = 4)
df_transformed = pd.DataFrame(knn_imputer.fit_transform(df_pre_knn_mice), columns = df_pre_knn_mice.columns)

df_transformed[targets] = df_transformed[targets].round()

df_knn = init_df.copy()

df_knn[targets] = data_encoder.inverse_transform(df_transformed[targets])

In [32]:
hist(inds = df_knn, x_axis_list = ['fbs', 'restecg', 'exang', 'slope', 'thal'], title = 'Histograms of distributions of character variables using KNN', mode = 'hist', size = [2, 3])

####Iterative Imputer + Classifier

In [33]:
mice_imputer = IterativeImputer(estimator = RandomForestClassifier(random_state = 42), max_iter = 10, random_state = 42)

df_transformed = pd.DataFrame(mice_imputer.fit_transform(df_pre_knn_mice), columns=df_pre_knn_mice.columns)

# Округляем закодированные категории перед раскодированием
df_transformed[targets] = df_transformed[targets].round()

df_mice = init_df.copy()

# 3. Возвращаем текст обратно
df_mice[targets] = data_encoder.inverse_transform(df_transformed[targets])

In [34]:
hist(inds = df_mice, x_axis_list = ['fbs', 'restecg', 'exang', 'slope', 'thal'], title = 'Histograms of distributions of character variables using Iterative Imputer + Classifier', mode = 'hist', size = [2, 3])

In [35]:
data_filled_char = impute_mice(inds = init_df,
                           predictor_num_cols = ['age'],
                           predictor_char_cols = ['sex', 'dataset'],
                           target_cols = ['fbs', 'restecg', 'exang', 'slope', 'thal'],
                           target_type = 0)

MICE Imputation Accuracy for fbs:  89.19%
MICE Imputation Accuracy for restecg:  60.81%
MICE Imputation Accuracy for exang:  63.51%
MICE Imputation Accuracy for slope:  35.14%
MICE Imputation Accuracy for thal:  52.70%
MICE Imputation Average Accuracy:  60.27%


In [36]:
hist(inds = data_filled_char, x_axis_list = ['fbs', 'restecg', 'exang', 'slope', 'thal'], title = 'Histograms of distributions of character variables using MICE imputation', mode = 'hist', size = [2, 3])

In [37]:
data_rdy = impute_mice(inds = data_filled_char,
                       predictor_num_cols = ['age'],
                       predictor_char_cols = ['sex', 'dataset'],
                       target_cols = ['trestbps', 'chol', 'thalch', 'oldpeak', 'ca'],
                       target_type = 1)

MICE Imputation Root Mean Absolute Error for trestbps:  12.5321
MICE Imputation Root Mean Absolute Error for chol:  39.7928
MICE Imputation Root Mean Absolute Error for thalch:  17.7998
MICE Imputation Root Mean Absolute Error for oldpeak:  0.8363
MICE Imputation Root Mean Absolute Error for ca:  0.7162
MICE Imputation Average Root Mean Absolute Error:  14.3355
Root Mean Absolute Error is:  3.7862


In [38]:
hist(inds = data_rdy, x_axis_list = ['trestbps', 'chol', 'thalch', 'oldpeak', 'ca'], title = 'Histograms of distributions of numeric variables using MICE imputation', size = [2, 3])

In [39]:
# Print ratio of missing data in the columns
for i in data_rdy.columns:
    null_rate = round(data_rdy[i].isna().sum() / len(data_rdy) * 100, 1)
    if null_rate > 0 :
        print(f"{i} null rate: {null_rate}%")
    else:
      print(f"{i} does not contain missing values")

id does not contain missing values
age does not contain missing values
sex does not contain missing values
dataset does not contain missing values
cp does not contain missing values
trestbps does not contain missing values
chol does not contain missing values
fbs does not contain missing values
restecg does not contain missing values
thalch does not contain missing values
exang does not contain missing values
oldpeak does not contain missing values
slope does not contain missing values
ca does not contain missing values
thal does not contain missing values
num does not contain missing values


##Correlation Matrix

In [40]:
char_features = ['sex', 'dataset', 'fbs', 'restecg', 'exang', 'slope', 'thal', 'cp']
num_features = ['age', 'trestbps', 'chol', 'thalch', 'oldpeak', 'ca']
all_features = char_features + num_features

In [41]:
corr_matrix = calculate_global_correlation(inds = data_rdy,
                                           all_cols = all_features,
                                           char_cols = char_features,
                                           num_cols = num_features)

In [42]:
fig = px.imshow(
    corr_matrix,
    text_auto = ".2f",
    color_continuous_scale = "RdBu_r",
    zmin = -1.0,
    zmax = 1.0,
    labels=dict(x = "Variable X",
                y = "Variable Y",
                color = "Correlation"),
    x = corr_matrix.columns,
    y = corr_matrix.index,
    title = "Correlation Matrix"
)

fig.update_layout(
    title_x = 0.5,
    width = 950,
    height = 850,
    template = "plotly_white",
)

fig.update_xaxes(tickangle=45)

fig.show()

#Modeling

##Dataset Preparation

In [43]:
X = data_rdy.drop('num', axis=1)
y = data_rdy['num']

# Standard Scaler + One-Hot Encoder
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), num_features),
        ('cat', OneHotEncoder(drop='first', handle_unknown='ignore'), char_features)
    ]
)

# Transform Data
X = preprocessor.fit_transform(X)

# Feature's names
cat_encoder = preprocessor.named_transformers_['cat']
encoded_cat_features = cat_encoder.get_feature_names_out(char_features).tolist()
feature_names = num_features + encoded_cat_features
all_features_intercept = ['Intercept'] + feature_names

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.15, random_state = 42)

##Baseline models

###Logistic Regression

####Simple Example

In [47]:
logregr_model = multinomial_logistic_regression(
    X_train = X_train,
    y_train = y_train,
    X_test = X_test,
    y_test = y_test
)

         TRAINING PHASE: MULTINOMIAL LOGISTIC STATISTICAL REPORT        

--- Coefficients for Class 0 vs All Other Classes ---
                 Feature Estimate (Coef) Std Error   z-score p-value
               Intercept          1.4778    0.4963  2.977259  0.0029
                     age          0.1534    0.1389  1.104492  0.2694
                trestbps         -0.0785    0.1088 -0.721438  0.4706
                    chol         -0.0976    0.1409 -0.692503  0.4886
                  thalch          0.2462    0.1334  1.845799  0.0649
                 oldpeak         -0.6524    0.1270 -5.138977 <0.0001
                      ca         -0.6832    0.1371 -4.983955 <0.0001
                sex_Male         -0.9273    0.2839 -3.266516  0.0011
         dataset_Hungary          2.0130    0.2932  6.865614 <0.0001
     dataset_Switzerland         -1.6985    0.4203 -4.041372 <0.0001
   dataset_VA Long Beach          0.2101    0.3074  0.683346  0.4944
                fbs_True         -0.1070    

###KNN

In [48]:
knn_model = knn_classifier(
    X_train = X_train,
    y_train = y_train,
    X_test = X_test,
    y_test = y_test
)

                      KNN CLASSIFIER EVALUATION REPORT                   
Test Accuracy: 0.5145
Test Log Loss: 6.0871
Derived Log Loss R^2: -3.5403
-------------------------------------------------------------------------
Detailed Classification Matrix Metrics:
              precision    recall  f1-score   support

           0       0.67      0.85      0.75        60
           1       0.35      0.35      0.35        40
           2       0.29      0.25      0.27        16
           3       0.29      0.11      0.16        18
           4       0.00      0.00      0.00         4

    accuracy                           0.51       138
   macro avg       0.32      0.31      0.31       138
weighted avg       0.46      0.51      0.48       138



###Gradient Boosting

In [53]:
gb_model = gradient_boosting_classifier(
    X_train = X_train,
    y_train = y_train,
    X_test = X_test,
    y_test = y_test
)

                 GRADIENT BOOSTING EVALUATION REPORT                    
Test Accuracy: 0.6014
Test Log Loss: 1.6792
Derived Log Loss R^2: -0.2525
-------------------------------------------------------------------------
Detailed Classification Matrix Metrics:
              precision    recall  f1-score   support

           0       0.70      0.90      0.79        60
           1       0.53      0.45      0.49        40
           2       0.33      0.25      0.29        16
           3       0.47      0.39      0.42        18
           4       0.00      0.00      0.00         4

    accuracy                           0.60       138
   macro avg       0.41      0.40      0.40       138
weighted avg       0.56      0.60      0.57       138



###Decision Tree Classifier

In [55]:
dt_model = decision_tree_classifier(
    X_train = X_train,
    y_train = y_train,
    X_test = X_test,
    y_test = y_test
)

                   DECISION TREE EVALUATION REPORT                       
Test Accuracy: 0.5072
Test Log Loss: 17.0191
Derived Log Loss R^2: -11.6943
-------------------------------------------------------------------------
Detailed Classification Matrix Metrics:
              precision    recall  f1-score   support

           0       0.69      0.78      0.73        60
           1       0.42      0.40      0.41        40
           2       0.20      0.19      0.19        16
           3       0.27      0.22      0.24        18
           4       0.00      0.00      0.00         4

    accuracy                           0.51       138
   macro avg       0.32      0.32      0.32       138
weighted avg       0.48      0.51      0.49       138



###Random Forest

In [57]:
rf_model = random_forest_classifier(
    X_train = X_train,
    y_train = y_train,
    X_test = X_test,
    y_test = y_test
)

                   RANDOM FOREST EVALUATION REPORT                       
Test Accuracy: 0.5362
Test Log Loss: 0.9915
Derived Log Loss R^2: 0.2604
-------------------------------------------------------------------------
Detailed Classification Matrix Metrics:
              precision    recall  f1-score   support

           0       0.67      0.88      0.76        60
           1       0.38      0.28      0.32        40
           2       0.32      0.38      0.34        16
           3       0.36      0.22      0.28        18
           4       0.00      0.00      0.00         4

    accuracy                           0.54       138
   macro avg       0.35      0.35      0.34       138
weighted avg       0.49      0.54      0.50       138



###XGBoost

In [63]:
xgboost_model = xgboost_classifier(
    X_train = X_train,
    y_train = y_train,
    X_test = X_test,
    y_test = y_test
)

                      XGBOOST EVALUATION REPORT                          
Test Accuracy: 0.5580
Test Log Loss: 1.0270
Derived Log Loss R^2: 0.2340
-------------------------------------------------------------------------
Detailed Classification Matrix Metrics:
              precision    recall  f1-score   support

           0       0.69      0.88      0.77        60
           1       0.41      0.33      0.36        40
           2       0.30      0.19      0.23        16
           3       0.42      0.44      0.43        18
           4       0.00      0.00      0.00         4

    accuracy                           0.56       138
   macro avg       0.36      0.37      0.36       138
weighted avg       0.51      0.56      0.52       138



###AdaBoost

In [65]:
adaboost_model = adaboost_classifier(
    X_train = X_train,
    y_train = y_train,
    X_test = X_test,
    y_test = y_test
)

                      ADABOOST EVALUATION REPORT                         
Test Accuracy: 0.5072
Test Log Loss: 1.5759
Derived Log Loss R^2: -0.1754
-------------------------------------------------------------------------
Detailed Classification Matrix Metrics:
              precision    recall  f1-score   support

           0       0.63      0.83      0.72        60
           1       0.32      0.35      0.33        40
           2       0.00      0.00      0.00        16
           3       0.40      0.33      0.36        18
           4       0.00      0.00      0.00         4

    accuracy                           0.51       138
   macro avg       0.27      0.30      0.28       138
weighted avg       0.42      0.51      0.46       138



##Model Tuning

In [68]:
tuned_estimators, metrics_leaderboard = model_tuning(
    X_train = X_train,
    y_train = y_train,
    X_test = X_test,
    y_test = y_test,
    feature_names = feature_names,
    all_features_intercept = all_features_intercept
)

                 STARTING HYPERPARAMETER FINE-TUNING                     

Tuning Logistic Regression via GridSearchCV...
---------------------------------------------------------
 METRICS SUMMARY: LOGISTIC REGRESSION
---------------------------------------------------------
  Optimized Hyperparameters : {'C': 0.1, 'penalty': 'l2'}
  5-Fold CV Accuracy (Mean) : 0.5921 (±0.0234)
  Test Accuracy             : 0.5507
  Test Log Loss             : 1.0009
  Log Loss R^2 Score        : 0.2535

Detailed Classification Matrix (Test Data):
              precision    recall  f1-score   support

           0       0.66      0.88      0.76        60
           1       0.37      0.35      0.36        40
           2       0.43      0.19      0.26        16
           3       0.46      0.33      0.39        18
           4       0.00      0.00      0.00         4

    accuracy                           0.55       138
   macro avg       0.38      0.35      0.35       138
weighted avg       0.50      

##